# 06 - Incremental models

**You will learn**: `materialized='incremental'`, `is_incremental()`, `unique_key`, the `merge` strategy and `--full-refresh`.

**You will build**: `fct_sales` loaded incrementally.

Until now every `dbt run` rebuilds each table from scratch. With millions of rows that becomes slow and costly.
An **incremental model** only processes what is **new or changed** since the last run and merges it into the existing table.

Our source sends a new batch of orders every week, and sometimes **changes an old order** (about 2% of the orders of the previous batch become `returned`).
The model must handle both: new rows *and* updated rows.

In [ ]:
from helpers import *

## 1. Where do we stand?

> **This notebook needs a new batch of bronze data**, loaded by your trainer partway through (section 4 below). Nothing to
> prepare now - just know that after you convert `fct_sales` to incremental and run it once, you will wait for your trainer's
> announcement before continuing.

In [ ]:
before = q(f'''
    SELECT COUNT(*) AS rows, COUNT_IF(order_status = 'returned') AS returned_lines,
           MAX(order_updated_at) AS max_updated_at
    FROM gold.{SCHEMA}.fct_sales''')
batch_before = int(scalar(f"SELECT MAX(_batch_id) FROM bronze.sports_shop.sales_orders"))
display(before)
display(bronze_status())

## 2. Turn `fct_sales` into an incremental model

Edit `models/gold/fct_sales.sql` in two places.

**a) A config block at the very top:**

```sql
{{
    config(
        materialized='incremental',
        incremental_strategy='merge',
        unique_key='order_line_id',
        on_schema_change='append_new_columns'
    )
}}
```

* `incremental_strategy='merge'` uses `MERGE INTO`: matching rows (same `unique_key`) are updated, new ones inserted.
* `unique_key='order_line_id'` says how to match rows.
* `on_schema_change` tells dbt what to do if you add a column later.

**b) A filter on the orders CTE**, so that only orders changed since the last run are read:

```sql
with orders as (

    select * from {{ ref('stg_sales_orders') }}
    {% if is_incremental() %}
    where updated_at > (select max(order_updated_at) from {{ this }})
    {% endif %}

),
```

* `is_incremental()` is true only when the table already exists and we are **not** doing a full refresh.
* `{{ this }}` is the table being built.
* Because a returned order gets a new `updated_at`, all its lines are picked up again and **merged**.

Do the edit in the editor, then continue.

In [ ]:
src = (SRC / "models/gold/fct_sales.sql").read_text(encoding="utf-8")
check("config block added", "materialized='incremental'" in src or 'materialized="incremental"' in src)
check("is_incremental filter added", "is_incremental()" in src)

## 3. First run: full refresh

The table exists as a plain table, so we rebuild it once as an incremental model with `--full-refresh`.

In [ ]:
dbt("run --select fct_sales --full-refresh")

A second run with **no new data** should process nothing new. Look at the log line (`MERGE`) and the row count:

In [ ]:
dbt("run --select fct_sales")
q(f"SELECT COUNT(*) AS rows FROM gold.{SCHEMA}.fct_sales")

Let us see the SQL dbt actually ran. This is the `MERGE`:

In [ ]:
show_file("target/run/alpsport/src/models/gold/fct_sales.sql")

## 4. A new batch arrives

### Your trainer loads the next batch

The bronze data is **shared by the whole class**, so it is not loaded by you: your **trainer** runs the generator notebook once
(mode `append`, it adds one week of orders (about 600), 40 new customers and some price changes, and turns ~2% of the previous batch's orders into `returned`) and announces it. Nothing to do until then, except re-running the next cell every now and then.

**Wait for the announcement** of your trainer before running the next cell.

In [ ]:
# Re-run this cell until your trainer has announced the new batch
status = bronze_status()
check("a new batch is available in bronze", int(status.last_batch[0]) > batch_before,
      "not yet: wait for your trainer's announcement, then re-run this cell")
status

`last_batch` is now higher than before and `last_order_date` moved into January 2026. Rebuild the pipeline:

In [ ]:
dbt("build")

In [ ]:
after = q(f'''
    SELECT COUNT(*) AS rows, COUNT_IF(order_status = 'returned') AS returned_lines,
           MAX(order_updated_at) AS max_updated_at
    FROM gold.{SCHEMA}.fct_sales''')
display(before)
display(after)
check("new lines were added", after.rows[0] > before.rows[0])
check("some lines changed to 'returned' (updated by the merge)", after.returned_lines[0] > before.returned_lines[0])

The row count grew (new orders) **and** some existing lines changed status: the merge updated them in place.
Check that the incremental result equals a full rebuild, that is the reconciliation test in `tests/` (it passed during `dbt build`).

## 5. Pitfalls

| Problem | Remedy |
|---|---|
| Rows that arrive **late** with an old timestamp are missed | subtract a lookback window in the filter, or use a load timestamp (`_ingested_at`) |
| A bug fixed in the logic does not change old rows | run with `--full-refresh` (schedule it from time to time) |
| Missing `unique_key` | duplicates: rows are only appended |
| Filtering with `>=` | reprocesses the last row every run (harmless with a merge, wasteful otherwise) |
| Changing the model schema | choose `on_schema_change` deliberately |

**`merge` vs `append`.** We used `incremental_strategy='merge'` because rows can change after the fact (a returned order).
If your source only ever adds rows and never updates old ones, `incremental_strategy='append'` is cheaper: it skips the
match-and-update logic and just inserts whatever the filtered query returns. The trade-off is that `append` blindly inserts -
if `is_incremental()` filter is not strict enough, or a row is processed twice, you get duplicates, since there's no `unique_key`
match to fall back on.

## Recap

* `incremental` + `unique_key` + `is_incremental()` filter = process only the delta.
* `merge` updates existing rows, inserts new ones.
* `--full-refresh` rebuilds from scratch.

---

In [ ]:
# restore_checkpoint(6)